# Prompt Optimization Techniques
### Practice Notebook

**Assumed pre-installed libraries:** none required (pure Python standard
library).

This notebook has no live LLM calls wired in -- every "model output" below
is a hand-written stand-in, so the focus stays entirely on the *process* of
iterative refinement, A/B testing, and versioning, which works identically
whether the outputs come from a stub function or a real model.


## 1. Iterative prompt refinement

The core loop: write a prompt, run it against representative test inputs,
find a specific failure, make ONE targeted change, re-test. Let's simulate
three rounds of refining a summarization prompt.


In [ ]:
test_articles = [
    "The city council approved a new budget for road repairs, allocating "
    "Rs.50 crore to fix potholes across 12 wards over the next six months.",
    "Scientists at a local university published a study showing that a "
    "new battery material could double electric vehicle range by 2027, "
    "though mass production is still several years away.",
]

def stub_model_v1(prompt: str, article: str) -> str:
    """# TODO: replace with a real LLM call. This stub simulates what a
    vague 'summarize this' prompt tends to produce: overly long, no fixed
    structure, inconsistent length across runs.
    """
    return f"This article talks about several things related to: {article[:80]}... " \
           f"and goes on to discuss further implications and background context " \
           f"at some length before concluding with related considerations."

prompt_v1 = "Summarize this article."
print("=== Prompt v1 ===")
print(prompt_v1)
for a in test_articles:
    print("\nOutput:", stub_model_v1(prompt_v1, a))


**Observed failure:** the v1 output is vague, doesn't actually extract the
specific facts (budget amount, ward count), and gives no guarantee of a
consistent length. Round 1 fix: add explicit constraints.


In [ ]:
def stub_model_v2(prompt: str, article: str) -> str:
    """# TODO: replace with a real LLM call. Simulates adding a length
    constraint and asking for key facts -- output is now shorter and more
    concrete, but still has no fixed structure across different articles.
    """
    if "budget" in article.lower():
        return "The city council approved Rs.50 crore for road repairs across 12 wards over 6 months."
    return "Researchers found a new battery material that could double EV range by 2027, pending mass production."

prompt_v2 = "Summarize this article in exactly one sentence, including the most important number mentioned."
print("=== Prompt v2 ===")
print(prompt_v2)
for a in test_articles:
    print("\nOutput:", stub_model_v2(prompt_v2, a))


**Observed improvement:** both outputs are now concrete and roughly the
same length. But notice they still differ in *structure* -- one leads with
the actor ("the city council"), the other leads with the finding
("researchers found"). If a downstream system needs a consistent structure
(e.g., to always start with the key number), a further round is needed.

**Exercise 3.1:** Write a `stub_model_v3` and `prompt_v3` that additionally
enforces a fixed structure, e.g., "Start your sentence with the single most
important number, then explain what it refers to." Test it against both
articles and confirm the structure is now consistent.


## 2. A/B testing prompts

When two prompt variants both seem reasonable, compare them systematically
against the same test set using a rubric (Day 1) rather than trusting a
gut feeling from one or two examples.


In [ ]:
import random

def stub_variant_A(article: str) -> str:
    """# TODO: replace with a real LLM call using prompt variant A."""
    random.seed(hash(article) % 1000)
    return random.choice([
        "Good, concise summary with the key number included.",
        "Good, concise summary with the key number included.",
        "Slightly vague summary, number is present but buried.",
    ])

def stub_variant_B(article: str) -> str:
    """# TODO: replace with a real LLM call using prompt variant B."""
    random.seed((hash(article) + 1) % 1000)
    return random.choice([
        "Excellent summary, leads with the number as required.",
        "Excellent summary, leads with the number as required.",
        "Excellent summary, leads with the number as required.",
        "Slightly vague summary, number is present but buried.",
    ])

def score_output(output_text: str) -> int:
    """Toy rubric scorer standing in for a real Day 1-style rubric score.
    # TODO: replace with a real rubric-based or LLM-as-judge score.
    """
    if "excellent" in output_text.lower() or "good, concise" in output_text.lower():
        return 5
    return 2

def ab_test(variant_a_fn, variant_b_fn, test_inputs: list) -> dict:
    a_wins, b_wins, ties = 0, 0, 0
    for inp in test_inputs:
        score_a = score_output(variant_a_fn(inp))
        score_b = score_output(variant_b_fn(inp))
        if score_a > score_b:
            a_wins += 1
        elif score_b > score_a:
            b_wins += 1
        else:
            ties += 1
    n = len(test_inputs)
    return {"a_win_rate": a_wins / n, "b_win_rate": b_wins / n,
            "tie_rate": ties / n, "n": n}

# A slightly larger test set than just the 2 articles above, to make the
# win-rate comparison a bit more meaningful (see Exercise 3.2).
larger_test_set = test_articles * 5   # repeated for demo purposes only

results = ab_test(stub_variant_A, stub_variant_B, larger_test_set)
print(results)


**Exercise 3.2 (important statistical caution):** Re-run the cell above with
`larger_test_set = test_articles` (just 2 items) instead of `test_articles *
5`. Does the win rate change? With only 2 test cases, how confident should
you be that one prompt variant is genuinely better, versus the result being
noise? What is a more trustworthy way to decide "prompt B is better" than a
win rate computed from a handful of examples? (Hint: think about how many
test cases -- and how *representative* of real usage those cases are --
you'd want before trusting a win-rate difference.)


## 3. Prompt versioning basics

Treat prompts like code: keep a record of every version, what changed, why,
and how it scored -- not just the current "final" prompt with no history.


In [ ]:
from datetime import date

prompt_registry = []

def log_prompt_version(version: str, prompt_text: str, change_rationale: str,
                        eval_score: float):
    prompt_registry.append({
        "version": version,
        "prompt_text": prompt_text,
        "change_rationale": change_rationale,
        "eval_score": eval_score,
        "date": str(date.today()),
    })

log_prompt_version("v1", "Summarize this article.",
                    "Initial baseline prompt.", eval_score=2.0)
log_prompt_version("v2", "Summarize this article in exactly one sentence, "
                          "including the most important number mentioned.",
                    "v1 outputs were vague and inconsistent length; added "
                    "explicit length and content constraints.", eval_score=3.8)
log_prompt_version("v3", "Summarize this article in exactly one sentence. "
                          "Start with the single most important number, "
                          "then explain what it refers to.",
                    "v2 outputs had inconsistent structure across articles; "
                    "added an explicit structural constraint.", eval_score=4.6)

for entry in prompt_registry:
    print(f"[{entry['version']}] score={entry['eval_score']}  "
          f"({entry['date']})")
    print(f"   prompt: {entry['prompt_text']}")
    print(f"   why:    {entry['change_rationale']}\n")


**Exercise 3.3 (mini deliverable):** Extend `prompt_registry` with a `v4`
entry that intentionally makes things *worse* (e.g., an overly restrictive
constraint that breaks on one of your test articles), with an honest
`change_rationale` and a lower `eval_score`. This is realistic -- not every
iteration is an improvement -- and the whole point of versioning is being
able to identify this and roll back to `v3`, which you couldn't do if you
had only kept the current prompt with no history.


## 4. Common failure patterns and how to fix them

| Failure pattern | Symptom | Fix |
|---|---|---|
| Vague instructions | Inconsistent, unpredictable output | Add explicit output format, constraints, success criteria |
| Missing context | Model guesses at background it doesn't have | Add relevant context or few-shot examples directly in the prompt |
| Overloaded prompt | Quality drops on every sub-task when asked to do too much at once | Split into a prompt chain (Week 3, Day 3) |
| Format drift | Output structure changes across repeated runs | Explicit formatting instructions, structured output (e.g. JSON schema), or few-shot examples |
| Leading questions | Model agrees with a false premise instead of correcting it | Instruct the model to verify premises; test explicitly against adversarial/leading inputs |

Let's build a tiny detector for one of these patterns: format drift.


In [ ]:
def stub_model_format_drift(article: str, run_number: int) -> str:
    """# TODO: replace with a real LLM call. Simulates a prompt that
    doesn't pin down output format -- the SAME input produces differently
    STRUCTURED outputs across repeated runs.
    """
    formats = [
        f"Summary: {article[:40]}...",
        f"1. {article[:40]}...",
        f"Here's what happened: {article[:40]}...",
    ]
    return formats[run_number % len(formats)]

def detect_format_drift(model_fn, article: str, n_runs: int = 3) -> bool:
    """Runs the same input multiple times and checks whether the output's
    STARTING PATTERN (a crude proxy for structure) stays consistent.
    """
    outputs = [model_fn(article, i) for i in range(n_runs)]
    starts = [o.split(":")[0] if ":" in o else o[:10] for o in outputs]
    drift_detected = len(set(starts)) > 1
    return drift_detected, outputs

drift, outputs = detect_format_drift(stub_model_format_drift, test_articles[0])
print("Format drift detected:", drift)
for o in outputs:
    print(" ", o)


**Exercise 3.4 (mini deliverable):** Write a `prompt_fix` string that
explicitly pins down the output format (e.g., "Always start your response
with the exact text 'Summary:' followed by a single sentence."), and write
a corresponding `stub_model_fixed` function that always follows it
regardless of `run_number`. Confirm `detect_format_drift` now reports
`False` across at least 5 runs.

**Exercise 3.5 (capstone for this notebook):** Pick one prompt from your own
Week 2 or Week 3 practice notebooks. Run it (or trace through it manually)
against 3 different inputs, identify one concrete failure pattern from the
table above, and write out a full `prompt_registry`-style log (at least 2
versions) documenting your fix, following the format from Part 3.
